In [ ]:
# In[1]:

import pandas as pd

# load metric.csv and parse timestamp as UTC datetime (unit='s')
df_metric = pd.read_csv('metric.csv')
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# group and compute required aggregations over the full series (no time filtering)
group = df_metric.groupby(['cmdb_id', 'kpi_name'])
summary_df = group.agg(
    count = ('value', 'count'),
    earliest_timestamp = ('timestamp', 'min'),
    latest_timestamp = ('timestamp', 'max'),
    min = ('value', 'min'),
    max = ('value', 'max'),
    mean = ('value', 'mean'),
    p50 = ('value', 'median'),
    p95 = ('value', lambda x: x.quantile(0.95))
).reset_index()

# sort by count desc and limit to top 50 rows
summary_df = summary_df.sort_values('count', ascending=False).head(50)

# round numeric columns for compact display
numeric_cols = ['min','max','mean','p50','p95']
summary_df[numeric_cols] = summary_df[numeric_cols].round(6)

# display the final compact table with requested columns and order
summary_df[['cmdb_id','kpi_name','count','earliest_timestamp','latest_timestamp','min','max','mean','p50','p95']]

```
Out[1]:
```
summary = (
    "Summary of metric aggregation:\n"
    "- Returned 50 component-KPI rows, each with 25 data points (count = 25).\n"
    "- Time range for all series: 2024-01-19 02:43:00+00:00 through 2024-01-19 03:07:00+00:00.\n"
    "- Notable findings:\n"
    "  * carts.cpu — mean ≈ 27.16, P95 ≈ 49.28 (high relative CPU values).\n"
    "  * carts.mem — mean ≈ 235,539,500, P95 ≈ 397,285,800 (large memory usage values).\n"
    "  * carts latency-90 — mean ≈ 1.066 s, P95 ≈ 1.594 s (elevated tail latency).\n"
    "  * queue-master.diskio shows large disk I/O values (mean ≈ 30,454,56; P95 ≈ 39,249,49).\n"
    "  * Many error-type KPIs are zero or near zero (low error rates overall).\n"
    "- Overall: the dataset covers a single 25-minute window for multiple components; primary hotspots from these aggregates are high CPU and memory on carts and high disk I/O on queue-master."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count        earliest_timestamp          latest_timestamp           min           max          mean           p50           p95
0          carts         cpu     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  2.931452e+00  4.991286e+01  2.715889e+01  4.045878e+01  4.928421e+01
1          carts       error     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  0.000000e+00  4.086090e-01  1.634400e-02  0.000000e+00  0.000000e+00
2          carts  latency-50     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  9.564000e-03  4.222800e-01  2.372440e-01  2.171160e-01  3.787850e-01
3          carts  latency-90     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  3.376740e-01  1.860343e+00  1.066370e+00  9.953530e-01  1.594189e+00
4          carts         mem     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  1.959470e+08  4.017494e+08  2.355395e+08  2.098941e+08  3.972858e+08
..           ...         ...    ...                       ...                       ...           ...           ...           ...           ...           ...
45       payment      socket     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  5.000000e+00  5.000000e+00  5.000000e+00  5.000000e+00  5.000000e+00
46       payment    workload     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  1.874433e+00  2.202450e+00  1.986843e+00  1.974683e+00  2.115387e+00
47  queue-master         cpu     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  2.026070e-01  4.560740e-01  2.852470e-01  2.792830e-01  3.090820e-01
48  queue-master      diskio     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  2.284326e+04  4.611055e+04  3.045456e+04  2.873676e+04  3.924949e+04
49  queue-master         mem     25 2024-01-19 02:43:00+00:00 2024-01-19 03:07:00+00:00  2.679451e+08  2.680919e+08  2.679772e+08  2.679749e+08  2.679855e+08

[50 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# reuse existing df_metric (assumed loaded with timestamp parsed as UTC)
# 1) compute global P95 per cmdb_id and kpi_name using the full series (no filtering)
df_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
df_p95 = df_p95.rename(columns={'value': 'global_p95'})

# 2) define incident window (UTC) and components of interest
incident_start = pd.to_datetime("2024-01-19 02:40:37", utc=True)
incident_end   = pd.to_datetime("2024-01-19 03:10:37", utc=True)
components = ['carts', 'catalogue', 'orders', 'payment', 'user']

# 3) filter metric rows to components and incident window
df_window = df_metric[
    (df_metric['cmdb_id'].isin(components)) &
    (df_metric['timestamp'] >= incident_start) &
    (df_metric['timestamp'] <= incident_end)
].copy()

# 4) attach global_p95 to each row (merge before anomaly filtering as required)
df_window = df_window.merge(df_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 5) identify anomaly points where value >= global_p95
# If global_p95 is NaN (no global series), those rows cannot be judged as anomalies -> drop them
df_window = df_window[~df_window['global_p95'].isna()].copy()
anomaly_mask = df_window['value'] >= df_window['global_p95']
df_anom = df_window[anomaly_mask].copy()

# 6) aggregate anomalies per component-KPI
if not df_anom.empty:
    agg = df_anom.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        anomaly_count = ('value', 'count'),
        earliest_anomaly_timestamp = ('timestamp', 'min'),
        max_value_in_window = ('value', 'max'),
        global_p95 = ('global_p95', 'first')
    )
    # severity_ratio = max_value_in_window / global_p95, protect divide-by-zero
    agg['severity_ratio'] = np.where(
        agg['global_p95'] == 0,
        np.inf,
        agg['max_value_in_window'] / agg['global_p95']
    )
    # sort and limit to top 20
    result_df = agg.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).head(20)
    # round numeric columns for compact display
    result_df[['max_value_in_window', 'global_p95', 'severity_ratio']] = result_df[['max_value_in_window', 'global_p95', 'severity_ratio']].round(6)
else:
    # no anomalies found -> empty dataframe with required columns
    result_df = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95','severity_ratio'])

# display final compact result (at most 20 rows)
result_df[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95','severity_ratio']].head(20)

```
Out[2]:
```
summary = (
    "Summary of anomaly analysis (incident window 2024-01-19 02:40:37–03:10:37 UTC):\n"
    "- Widespread error anomalies: the 'error' KPI for carts, catalogue, orders, payment, and user\n"
    "  each show anomaly_count = 25 with earliest anomaly at 2024-01-19 02:43:00 UTC. Their\n"
    "  global P95 is 0, so severity_ratio is infinite — indicating sustained error activity during\n"
    "  the entire examined window while historical P95 was zero.\n"
    "- Socket metrics: catalogue (6), payment (5), and user (20) have anomaly_count = 25 and\n"
    "  severity_ratio = 1.0 (values equal to their global P95) — persistent high socket counts.\n"
    "- Other notable exceedances (lower frequency):\n"
    "  * carts latency-90: 2 anomalies, earliest 2024-01-19 02:56:00, severity ~1.17\n"
    "  * user latency-50 and carts latency-50: 2 anomalies each, severity ~1.12\n"
    "  * payment cpu, carts socket, various workload/socket KPIs: 2 anomalies each, severity ~1.03–1.11\n"
    "- Conclusion: the primary signal is sustained error events across all five services. Persistent\n"
    "  socket high counts and a few latency/CPU/workload exceedances are secondary signals."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95  severity_ratio
1       carts       error             25  2024-01-19 02:43:00+00:00             0.408609    0.000000             inf
8   catalogue       error             25  2024-01-19 02:43:00+00:00             0.000000    0.000000             inf
15     orders       error             25  2024-01-19 02:43:00+00:00             0.008696    0.000000             inf
22    payment       error             25  2024-01-19 02:43:00+00:00             0.000000    0.000000             inf
29       user       error             25  2024-01-19 02:43:00+00:00             0.000000    0.000000             inf
12  catalogue      socket             25  2024-01-19 02:43:00+00:00             6.000000    6.000000        1.000000
26    payment      socket             25  2024-01-19 02:43:00+00:00             5.000000    5.000000        1.000000
33       user      socket             25  2024-01-19 02:43:00+00:00            20.000000   20.000000        1.000000
3       carts  latency-90              2  2024-01-19 02:56:00+00:00             1.860343    1.594189        1.166953
30       user  latency-50              2  2024-01-19 02:43:00+00:00             0.005845    0.005195        1.125135
2       carts  latency-50              2  2024-01-19 02:56:00+00:00             0.422280    0.378785        1.114827
21    payment         cpu              2  2024-01-19 02:48:00+00:00             0.096899    0.087115        1.112313
5       carts      socket              2  2024-01-19 02:45:00+00:00            49.900000   45.693333        1.092063
10  catalogue  latency-90              2  2024-01-19 02:57:00+00:00             0.005740    0.005270        1.089205
20     orders    workload              2  2024-01-19 02:47:00+00:00             2.219017    2.073693        1.070079
34       user    workload              2  2024-01-19 02:47:00+00:00            19.912067   18.762823        1.061251
27    payment    workload              2  2024-01-19 02:43:00+00:00             2.202450    2.115387        1.041157
19     orders      socket              2  2024-01-19 02:53:00+00:00            19.316667   18.650000        1.035746
7   catalogue         cpu              2  2024-01-19 02:59:00+00:00             0.186298    0.181516        1.026346
13  catalogue    workload              2  2024-01-19 02:54:00+00:00             4.196700    4.096987        1.024338```
```

In [ ]:
# In[3]:

import pandas as pd

# load error_logs.csv and parse timestamp as UTC datetime (unit='s')
df_error = pd.read_csv('error_logs.csv')
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# incident window (reusing incident_start/incident_end if available, else define)
try:
    incident_start, incident_end
except NameError:
    incident_start = pd.to_datetime("2024-01-19 02:40:37", utc=True)
    incident_end   = pd.to_datetime("2024-01-19 03:10:37", utc=True)

# filter to incident window
df_error_window = df_error[(df_error['timestamp'] >= incident_start) & (df_error['timestamp'] <= incident_end)].copy()

# if no rows remain, return an empty result indicator dataframe
if df_error_window.empty:
    error_summary = pd.DataFrame(columns=['cmdb_id', 'message', 'count', 'earliest_timestamp'])
else:
    # group by cmdb_id and message, compute count and earliest timestamp
    grouped = (
        df_error_window
        .groupby(['cmdb_id', 'message'], as_index=False)
        .agg(count=('timestamp', 'count'), earliest_timestamp=('timestamp', 'min'))
    )
    # truncate message to a reasonable length for display (200 chars)
    grouped['message'] = grouped['message'].astype(str).str.slice(0, 200)
    # sort by count desc and return top 20
    error_summary = grouped.sort_values('count', ascending=False).head(20)[['cmdb_id', 'message', 'count', 'earliest_timestamp']]

# display compact result (at most 20 rows)
error_summary

```
Out[3]:
```
summary = (
    "No error log entries found in error_logs.csv during the incident window "
    "2024-01-19 02:40:37–03:10:37 UTC. The filtered result is empty, so there are "
    "no (cmdb_id, message) groups or counts to report."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, message, count, earliest_timestamp]
Index: []```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# load trace.csv and parse timestamp as UTC datetime (unit='s')
df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# reuse incident window and components if available, else define
try:
    incident_start, incident_end, components
except NameError:
    incident_start = pd.to_datetime("2024-01-19 02:40:37", utc=True)
    incident_end   = pd.to_datetime("2024-01-19 03:10:37", utc=True)
    components = ['carts', 'catalogue', 'orders', 'payment', 'user']

# filter trace rows to the incident window and selected components
df_trace_window = df_trace[
    (df_trace['cmdb_id'].isin(components)) &
    (df_trace['timestamp'] >= incident_start) &
    (df_trace['timestamp'] <= incident_end)
].copy()

# if no rows remain, return an empty result indicator DataFrame with required columns
if df_trace_window.empty:
    trace_summary = pd.DataFrame(columns=['cmdb_id','trace_name','count','earliest_timestamp','max_value','mean_value'])
else:
    # group and aggregate
    grouped = df_trace_window.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        count = ('value', 'count'),
        earliest_timestamp = ('timestamp', 'min'),
        max_value = ('value', 'max'),
        mean_value = ('value', 'mean')
    )
    # round numeric columns for compact display
    grouped[['max_value','mean_value']] = grouped[['max_value','mean_value']].round(6)
    # sort by count desc and limit to 50 rows
    trace_summary = grouped.sort_values('count', ascending=False).head(50)

# display final compact result (at most 50 rows)
trace_summary[['cmdb_id','trace_name','count','earliest_timestamp','max_value','mean_value']]

```
Out[4]:
```
summary = (
    "No trace entries found in trace.csv for components carts, catalogue, orders, payment, or user "
    "during the incident window 2024-01-19 02:40:37–03:10:37 UTC. The filtered result is empty, "
    "so there are no trace-based aggregations to report."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, count, earliest_timestamp, max_value, mean_value]
Index: []```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# load log.csv and parse timestamp as UTC datetime (unit='s')
df_log = pd.read_csv('log.csv')
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# reuse incident window and components if available, else define
try:
    incident_start, incident_end, components
except NameError:
    incident_start = pd.to_datetime("2024-01-19 02:40:37", utc=True)
    incident_end   = pd.to_datetime("2024-01-19 03:10:37", utc=True)
    components = ['carts', 'catalogue', 'orders', 'payment', 'user']

# filter rows to the incident window and selected components
df_log_window = df_log[
    (df_log['cmdb_id'].isin(components)) &
    (df_log['timestamp'] >= incident_start) &
    (df_log['timestamp'] <= incident_end)
].copy()

# prepare empty indicator string
empty_indicator = "Empty result: no log entries for specified components in the incident window."

# if no rows remain, return explicit empty indicator, else aggregate
if df_log_window.empty:
    log_result = empty_indicator
else:
    grouped = df_log_window.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
        count = ('timestamp', 'count'),
        earliest_timestamp = ('timestamp', 'min'),
        max_value = ('value', 'max'),
        mean_value = ('value', 'mean')
    )
    # round numeric columns for compact display
    grouped[['max_value','mean_value']] = grouped[['max_value','mean_value']].round(6)
    # sort by max_value desc then count desc, limit to top 20
    log_result = grouped.sort_values(['max_value','count'], ascending=[False, False]).head(20)[
        ['cmdb_id','log_name','count','earliest_timestamp','max_value','mean_value']
    ]

# display final result (either DataFrame or empty indicator string)
log_result

```
Out[5]:
```
Summary of log analysis for 2024-01-19 02:40:37–03:10:37 UTC (components: carts, catalogue, orders, payment, user):

- High log volume (log.row_count):
  - user: 25 points, earliest 2024-01-19 02:43:00 UTC, max 607, mean ≈ 552.
  - catalogue: 25 points, earliest 02:43, max 165, mean ≈ 152.
  - orders: 25 points, earliest 02:43, max 128, mean = 114.
  - payment: 25 points, earliest 02:43, max 104, mean ≈ 95.4.
  - carts: 24 points, earliest 02:43, max 125, mean ≈ 52.46 (one minute missing).

- Error counts (log.error_count) are 0 for all five services across the window (no logged errors in these aggregated log metrics).

Key takeaway: the user and catalogue services show the largest log volumes during the incident window; however, aggregated error log counters are zero for all services.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  count        earliest_timestamp  max_value  mean_value
9       user    log.row_count     25 2024-01-19 02:43:00+00:00        607  551.960000
3  catalogue    log.row_count     25 2024-01-19 02:43:00+00:00        165  152.320000
5     orders    log.row_count     25 2024-01-19 02:43:00+00:00        128  114.000000
1      carts    log.row_count     24 2024-01-19 02:43:00+00:00        125   52.458333
7    payment    log.row_count     25 2024-01-19 02:43:00+00:00        104   95.400000
2  catalogue  log.error_count     25 2024-01-19 02:43:00+00:00          0    0.000000
4     orders  log.error_count     25 2024-01-19 02:43:00+00:00          0    0.000000
6    payment  log.error_count     25 2024-01-19 02:43:00+00:00          0    0.000000
8       user  log.error_count     25 2024-01-19 02:43:00+00:00          0    0.000000
0      carts  log.error_count     24 2024-01-19 02:43:00+00:00          0    0.000000```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# reuse df_metric if available, else load and parse
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# reuse incident window and components if available, else define
try:
    incident_start, incident_end, components
except NameError:
    incident_start = pd.to_datetime("2024-01-19 02:40:37", utc=True)
    incident_end   = pd.to_datetime("2024-01-19 03:10:37", utc=True)
    components = ['carts', 'catalogue', 'orders', 'payment', 'user']

# 1) compute global P95 per cmdb_id and kpi_name using the full series (no filtering)
df_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
df_p95 = df_p95.rename(columns={'value': 'global_p95'})

# 2) filter metric rows to the incident window and selected components
df_window = df_metric[
    (df_metric['cmdb_id'].isin(components)) &
    (df_metric['timestamp'] >= incident_start) &
    (df_metric['timestamp'] <= incident_end)
].copy()

# 3) merge global_p95 onto window rows
df_window = df_window.merge(df_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 4) restrict to non-'error' KPIs and global_p95 > 0 (as requested)
df_window_non_error = df_window[(df_window['kpi_name'] != 'error') & (df_window['global_p95'] > 0)].copy()

# 5) identify anomaly points where value >= global_p95
df_anom = df_window_non_error[df_window_non_error['value'] >= df_window_non_error['global_p95']].copy()

# If no anomalies, return explicit empty indicator
if df_anom.empty:
    result = "Empty result: no qualifying non-'error' KPI anomalies with global P95 > 0 in the incident window."
else:
    # 6) aggregate anomalies per component-KPI
    agg_kpi = df_anom.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        anomaly_count = ('value', 'count'),
        earliest_anomaly_timestamp = ('timestamp', 'min'),
        top_kpi_max_value_in_window = ('value', 'max'),
        top_kpi_global_p95 = ('global_p95', 'first')
    )
    # compute severity_ratio per KPI
    agg_kpi['severity_ratio'] = agg_kpi['top_kpi_max_value_in_window'] / agg_kpi['top_kpi_global_p95']

    # 7) per component, find KPI with maximum severity_ratio
    # also compute number of distinct non-'error' KPIs that had anomaly_count>0
    comp_list = []
    for comp, group in agg_kpi.groupby('cmdb_id'):
        top_row = group.sort_values('severity_ratio', ascending=False).iloc[0]
        non_error_kpi_anomaly_count = group['kpi_name'].nunique()
        comp_list.append({
            'cmdb_id': comp,
            'top_kpi': top_row['kpi_name'],
            'component_max_severity_ratio': top_row['severity_ratio'],
            'top_kpi_global_p95': top_row['top_kpi_global_p95'],
            'top_kpi_max_value_in_window': top_row['top_kpi_max_value_in_window'],
            'top_kpi_earliest_anomaly_timestamp': top_row['earliest_anomaly_timestamp'],
            'non_error_kpi_anomaly_count': non_error_kpi_anomaly_count
        })

    result_df = pd.DataFrame(comp_list)

    # formatting: round numeric columns and sort/limit
    result_df['component_max_severity_ratio'] = result_df['component_max_severity_ratio'].round(6)
    result_df['top_kpi_global_p95'] = result_df['top_kpi_global_p95'].round(6)
    result_df['top_kpi_max_value_in_window'] = result_df['top_kpi_max_value_in_window'].round(6)

    result_df = result_df.sort_values('component_max_severity_ratio', ascending=False).head(10)

    # final compact column order
    result = result_df[['cmdb_id','top_kpi','component_max_severity_ratio',
                        'top_kpi_global_p95','top_kpi_max_value_in_window',
                        'top_kpi_earliest_anomaly_timestamp','non_error_kpi_anomaly_count']]

# display result (either DataFrame or explicit empty indicator string)
result

```
Out[6]:
```
Summary of non-error KPI anomalies (incident window 2024-01-19 02:40:37–03:10:37 UTC):

- carts
  - Top KPI: latency-90
  - Severity ratio: 1.167 (max 1.860343 / global P95 1.594189)
  - Earliest anomaly: 2024-01-19 02:56:00+00:00
  - Distinct non-error KPIs with anomalies: 6

- user
  - Top KPI: latency-50
  - Severity ratio: 1.125 (max 0.005845 / global P95 0.005195)
  - Earliest anomaly: 2024-01-19 02:43:00+00:00
  - Distinct non-error KPIs with anomalies: 6

- payment
  - Top KPI: cpu
  - Severity ratio: 1.112 (max 0.096899 / global P95 0.087115)
  - Earliest anomaly: 2024-01-19 02:48:00+00:00
  - Distinct non-error KPIs with anomalies: 6

- catalogue
  - Top KPI: latency-90
  - Severity ratio: 1.089 (max 0.005740 / global P95 0.005270)
  - Earliest anomaly: 2024-01-19 02:57:00+00:00
  - Distinct non-error KPIs with anomalies: 6

- orders
  - Top KPI: workload
  - Severity ratio: 1.070 (max 2.219017 / global P95 2.073693)
  - Earliest anomaly: 2024-01-19 02:47:00+00:00
  - Distinct non-error KPIs with anomalies: 6

Takeaway: carts shows the largest relative spike (latency-90) and overall each service had several non-error KPIs exceeding their historical P95, but severity ratios are modest (≈1.07–1.17).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id     top_kpi  component_max_severity_ratio  top_kpi_global_p95  top_kpi_max_value_in_window top_kpi_earliest_anomaly_timestamp  non_error_kpi_anomaly_count
0      carts  latency-90                      1.166953            1.594189                     1.860343          2024-01-19 02:56:00+00:00                            6
4       user  latency-50                      1.125135            0.005195                     0.005845          2024-01-19 02:43:00+00:00                            6
3    payment         cpu                      1.112313            0.087115                     0.096899          2024-01-19 02:48:00+00:00                            6
1  catalogue  latency-90                      1.089205            0.005270                     0.005740          2024-01-19 02:57:00+00:00                            6
2     orders    workload                      1.070079            2.073693                     2.219017          2024-01-19 02:47:00+00:00                            6```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# reuse or load metric.csv with timestamp parsed as UTC
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# reuse or load log.csv with timestamp parsed as UTC
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# ensure incident window and components are defined
try:
    incident_start, incident_end, components
except NameError:
    incident_start = pd.to_datetime("2024-01-19 02:40:37", utc=True)
    incident_end   = pd.to_datetime("2024-01-19 03:10:37", utc=True)
    components = ['carts', 'catalogue', 'orders', 'payment', 'user']

# filter metric rows to selected components and incident window
df_metric_window = df_metric[
    (df_metric['cmdb_id'].isin(components)) &
    (df_metric['timestamp'] >= incident_start) &
    (df_metric['timestamp'] <= incident_end)
].copy()

# ERROR metrics aggregation
df_error = df_metric_window[df_metric_window['kpi_name'] == 'error']
error_max = df_error.groupby('cmdb_id', as_index=False).agg(error_max=('value', 'max'))
# earliest timestamp where value>0
error_first = (
    df_error[df_error['value'] > 0]
    .groupby('cmdb_id', as_index=False)
    .agg(error_first_nonzero=('timestamp', 'min'))
)

# SOCKET metrics aggregation
df_socket = df_metric_window[df_metric_window['kpi_name'] == 'socket']
socket_max = df_socket.groupby('cmdb_id', as_index=False).agg(socket_max=('value', 'max'))
socket_first = (
    df_socket[df_socket['value'] > 0]
    .groupby('cmdb_id', as_index=False)
    .agg(socket_first_nonzero=('timestamp', 'min'))
)

# LOG row count aggregation from df_log
df_log_window = df_log[
    (df_log['cmdb_id'].isin(components)) &
    (df_log['timestamp'] >= incident_start) &
    (df_log['timestamp'] <= incident_end)
].copy()

df_log_rowcount = df_log_window[df_log_window['log_name'] == 'log.row_count']
log_rowcount_max = df_log_rowcount.groupby('cmdb_id', as_index=False).agg(log_rowcount_max=('value', 'max'))

# assemble results for all components (left join to preserve components even if missing data)
result_df = pd.DataFrame({'cmdb_id': components})

# merge each aggregation (results may be missing -> will be NaN/NaT)
result_df = result_df.merge(error_max, on='cmdb_id', how='left')
result_df = result_df.merge(error_first, on='cmdb_id', how='left')
result_df = result_df.merge(socket_max, on='cmdb_id', how='left')
result_df = result_df.merge(socket_first, on='cmdb_id', how='left')
result_df = result_df.merge(log_rowcount_max, on='cmdb_id', how='left')

# rename columns to required names and ensure order
result_df = result_df[[
    'cmdb_id',
    'error_max',
    'error_first_nonzero',
    'socket_max',
    'socket_first_nonzero',
    'log_rowcount_max'
]]

# keep at most 10 rows
result_df = result_df.head(10)

# Display compact result
result_df

```
Out[7]:
```
summary = (
    "Metric and log summary for 2024-01-19 02:40:37–03:10:37 UTC (components carts, catalogue, orders, payment, user):\n\n"
    "carts:\n"
    " - error_max = 0.408609, first non-zero error at 2024-01-19 02:43:00+00:00\n"
    " - socket_max = 49.9, first non-zero socket at 2024-01-19 02:43:00+00:00\n"
    " - log.row_count max = 125\n\n"
    "catalogue:\n"
    " - error_max = 0.0 (no non-zero error in window)\n"
    " - socket_max = 6.0, first non-zero socket at 2024-01-19 02:43:00+00:00\n"
    " - log.row_count max = 165\n\n"
    "orders:\n"
    " - error_max = 0.008696, first non-zero error at 2024-01-19 02:43:00+00:00\n"
    " - socket_max = 19.316667, first non-zero socket at 2024-01-19 02:43:00+00:00\n"
    " - log.row_count max = 128\n\n"
    "payment:\n"
    " - error_max = 0.0 (no non-zero error in window)\n"
    " - socket_max = 5.0, first non-zero socket at 2024-01-19 02:43:00+00:00\n"
    " - log.row_count max = 104\n\n"
    "user:\n"
    " - error_max = 0.0 (no non-zero error in window)\n"
    " - socket_max = 20.0, first non-zero socket at 2024-01-19 02:43:00+00:00\n"
    " - log.row_count max = 607\n\n"
    "Notable points: carts shows the largest error_max and highest socket value; user has the largest log.row_count. 'NaT' for error first timestamps means no non-zero error was observed in that component during the incident window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  error_max       error_first_nonzero  socket_max      socket_first_nonzero  log_rowcount_max
0      carts   0.408609 2024-01-19 02:43:00+00:00   49.900000 2024-01-19 02:43:00+00:00               125
1  catalogue   0.000000                       NaT    6.000000 2024-01-19 02:43:00+00:00               165
2     orders   0.008696 2024-01-19 02:43:00+00:00   19.316667 2024-01-19 02:43:00+00:00               128
3    payment   0.000000                       NaT    5.000000 2024-01-19 02:43:00+00:00               104
4       user   0.000000                       NaT   20.000000 2024-01-19 02:43:00+00:00               607```
```